# Lección 14.1 — Agente, entorno, acción, recompensa: aprender a base de premios

> **Módulo 14 · Aprendizaje por refuerzo (introducción conceptual)** · Nivel 5 — Héroe
> ⏱️ Tiempo estimado: 80–100 min (lección con ejercicios intercalados ≈ 55 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 1.3 (tipos de aprendizaje), Lección 3.2 (función de pérdida)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con una analogía qué es el aprendizaje por refuerzo (*reinforcement learning*, RL) y en qué se diferencia del aprendizaje supervisado.
- Identificar el **agente**, el **entorno**, el **estado**, las **acciones** y la **recompensa** en cualquier problema.
- Calcular el **retorno** de un episodio y comparar políticas simulándolas.
- Reconocer el peligro de una recompensa mal diseñada (*reward hacking*).

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)   # semilla fija: así tus números coinciden con las respuestas
plt.rcParams["figure.figsize"] = (8, 4)
print("✅ Todo listo")

---
## 1. 📖 La intuición: enseñarle trucos a un perro

Nadie le explica a un perro con palabras qué es "sentarse". Lo que haces es:

```
 🐶 hace algo  ──►  tú reaccionas  ──►  🐶 ajusta lo que hará la próxima vez
 (se sienta)        (🦴 premio)          ("sentarme trae premios")
```

El perro **prueba** cosas, **observa** qué pasa y poco a poco repite lo que le trae premios y evita lo que no. Nadie le dio una tabla con "la respuesta correcta" para cada situación.

**El aprendizaje por refuerzo (RL, *reinforcement learning*) es exactamente esto, pero con un programa:** un **agente** aprende a **actuar** probando acciones y recibiendo **recompensas** (premios o castigos).

### ✋ Ejercicio 1 — Calentamiento
Piensa en cómo aprendiste a andar en bicicleta. ¿Alguien te dio una lista con "el ángulo correcto del manubrio" para cada instante? ¿Qué hizo el papel de "premio" y qué el de "castigo"?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Nadie te dio la respuesta correcta para cada instante (eso sería aprendizaje **supervisado**). Aprendiste **probando**: el castigo fue caerte o tambalearte y el premio, avanzar sin caer. Con muchos intentos, tu "política" (qué hacer en cada situación) mejoró. Eso es aprendizaje por refuerzo.
</details>

---
## 2. 📖 El concepto: el ciclo agente–entorno

Todo problema de RL tiene las mismas piezas:

```
            acción  aₜ
   ┌────────────────────────────┐
   │                            ▼
┌──┴─────┐                 ┌──────────┐
│ AGENTE │                 │ ENTORNO  │
└──▲─────┘                 └────┬─────┘
   │   nuevo estado sₜ₊₁        │
   └────── recompensa rₜ₊₁ ◄────┘
```

| Término | Qué es | En el perro |
|---|---|---|
| **Agente** | Quien decide y aprende | El perro |
| **Entorno** | Todo lo demás: el mundo con el que interactúa | Tú, la sala, la pelota |
| **Estado** (*state*) | La situación actual, lo que el agente "ve" | "Mi dueño tiene una galleta y dijo *siéntate*" |
| **Acción** (*action*) | Lo que el agente puede hacer | Sentarse, saltar, ladrar |
| **Recompensa** (*reward*) | Un número que dice qué tan bien le fue en ese paso | +1 galleta, 0 nada |
| **Política** (*policy*) | La "regla" del agente: qué acción elige en cada estado | "Si oigo *siéntate*, me siento" |
| **Episodio** | Una partida completa, del inicio a un final | Una sesión de entrenamiento |

El objetivo del agente **no** es la recompensa del próximo paso, sino **sumar la mayor recompensa posible a lo largo de todo el episodio**. A esa suma se le llama **retorno** (*return*).

### ✋ Ejercicio 2 — Encuentra las piezas
Para cada caso, identifica **agente, acciones y recompensa**:

| Caso | Agente | Acciones | Recompensa |
|---|---|---|---|
| Un programa que juega ajedrez | ? | ? | ? |
| Un termostato inteligente | ? | ? | ? |
| La app que elige qué video mostrarte | ? | ? | ? |

✍️ *Tu tabla:*

<details><summary><b>👀 Ver respuesta</b></summary>

| Caso | Agente | Acciones | Recompensa |
|---|---|---|---|
| Ajedrez | El programa | Mover una pieza | +1 si gana, −1 si pierde, 0 si empata (¡solo al final!) |
| Termostato | El controlador | Subir, bajar o mantener la calefacción | Comodidad de las personas menos el costo de energía |
| Videos | El sistema de recomendación | Qué video mostrar | Si lo ves, cuánto tiempo, si vuelves mañana |

Fíjate en el ajedrez: la recompensa llega **solo al final** de la partida. Saber cuál de las 40 jugadas fue la buena es uno de los grandes retos del RL (lo verás en la Sección 5).
</details>

### 📖 Nuestro primer entorno: el ratón en el pasillo 🐭

Un ratón está en un pasillo de 7 casillas. En un extremo hay **queso** 🧀 y en el otro, un **gato** 😾.

```
casilla:   0    1    2    3    4    5    6
          😾   ⬜   ⬜   🐭   ⬜   ⬜   🧀
```

- **Estado:** la casilla en la que está el ratón (0 a 6).
- **Acciones:** moverse a la izquierda (−1) o a la derecha (+1).
- **Recompensas:** +10 al llegar al queso, −10 al llegar al gato y −1 por cada otro paso (el ratón se cansa).
- **Fin del episodio:** al llegar al queso o al gato (o tras 50 pasos, para que nunca sea infinito).

La celda siguiente construye este entorno. No necesitas entender cada línea: solo fíjate en `paso()`, que es **el corazón de cualquier entorno de RL**: recibe una acción y devuelve *(nuevo estado, recompensa, ¿terminó?)*.

In [ ]:
class Pasillo:
    """Entorno: un pasillo con algo en cada extremo."""
    def __init__(self, largo=7, inicio=3, premio_izq=-10, premio_der=10, costo_paso=-1, max_pasos=50):
        self.largo, self.inicio = largo, inicio
        self.premio_izq, self.premio_der = premio_izq, premio_der
        self.costo_paso, self.max_pasos = costo_paso, max_pasos

    def reiniciar(self):
        self.pos, self.pasos = self.inicio, 0
        return self.pos

    def paso(self, accion):                      # accion: -1 (izquierda) o +1 (derecha)
        self.pos += accion
        self.pasos += 1
        if self.pos == 0:                        # llegó al extremo izquierdo
            return self.pos, self.premio_izq, True
        if self.pos == self.largo - 1:           # llegó al extremo derecho
            return self.pos, self.premio_der, True
        return self.pos, self.costo_paso, self.pasos >= self.max_pasos

    def dibujar(self):
        icono = lambda r: "🧀" if r > 0 else "😾"
        celdas = [icono(self.premio_izq)] + ["⬜"] * (self.largo - 2) + [icono(self.premio_der)]
        celdas[self.pos] = "🐭"
        return "".join(celdas)


def jugar_episodio(entorno, politica, mostrar=False):
    """Juega un episodio completo y devuelve el retorno (suma de recompensas)."""
    estado, retorno, terminado = entorno.reiniciar(), 0, False
    if mostrar: print("inicio      ", entorno.dibujar())
    while not terminado:
        accion = politica(estado)
        estado, recompensa, terminado = entorno.paso(accion)
        retorno += recompensa
        if mostrar: print(f"{'⬅️' if accion < 0 else '➡️'}  r = {recompensa:+4}", entorno.dibujar())
    if mostrar: print(f"Retorno del episodio: {retorno}")
    return retorno

pasillo = Pasillo()
print(pasillo.reiniciar(), pasillo.dibujar())

### 📖 Políticas: la "personalidad" del agente
Una **política** es simplemente una función: *estado → acción*. Probemos tres ratones con personalidades distintas.

In [ ]:
def siempre_derecha(estado):  return +1
def siempre_izquierda(estado): return -1
def al_azar(estado):           return rng.choice([-1, +1])

### 🔮 Ejercicio 3 — Calcula el retorno a mano
Antes de ejecutar, calcula en papel el **retorno** (la suma de recompensas) del ratón que va **siempre a la derecha** y del que va **siempre a la izquierda**. Recuerda: −1 por paso normal, +10 al llegar al queso, −10 al llegar al gato.

✍️ *Siempre derecha:* ___ · *Siempre izquierda:* ___

In [ ]:
jugar_episodio(pasillo, siempre_derecha, mostrar=True)
print()
jugar_episodio(pasillo, siempre_izquierda, mostrar=True);

<details><summary><b>👀 Ver respuesta</b></summary>

- **Siempre derecha:** −1 −1 +10 = **8**.
- **Siempre izquierda:** −1 −1 −10 = **−12**.

El último paso no cobra el −1: recibe directamente la recompensa del extremo.
</details>

### 🔮 Ejercicio 4 — ¿Y el ratón que se mueve al azar?
El ratón al azar llega a veces al queso y a veces al gato, y en el camino da muchos pasos de ida y vuelta. Como cada episodio es distinto, lo justo es **promediar muchos episodios**.

¿Su retorno promedio será positivo o negativo? ¿Mayor o menor que −12?

✍️ *Mi apuesta:*

In [ ]:
def retorno_promedio(entorno, politica, n=2000):
    return np.mean([jugar_episodio(entorno, politica) for _ in range(n)])

jugar_episodio(pasillo, al_azar, mostrar=True)
print(f"\nRetorno promedio en 2000 episodios al azar: {retorno_promedio(pasillo, al_azar):.1f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Ronda los **−8**: negativo, pero mejor que −12. Por simetría, llega al queso la mitad de las veces y al gato la otra mitad (los ±10 se cancelan en promedio), y en el camino da en promedio unos 9 pasos, que cuestan cerca de −8.

Moraleja: para comparar políticas que tienen azar **hay que simularlas muchas veces y promediar**, igual que en la Lección 4.6 promediabas varias divisiones de validación cruzada.
</details>

---
## 3. 📖 ¿En qué se diferencia del aprendizaje supervisado?

| | Supervisado (Módulos 4–5) | Por refuerzo |
|---|---|---|
| ¿Qué recibe? | Ejemplos con **la respuesta correcta** | Solo una **recompensa** (un número) después de actuar |
| ¿Sabe cuál era la mejor acción? | Sí, es la etiqueta | **No**: solo sabe qué tan bien le fue con la que eligió |
| ¿Cuándo llega la señal? | Inmediatamente, en cada ejemplo | A menudo **retrasada** (al final de la partida) |
| ¿Los datos dependen del modelo? | No: el dataset es fijo | **Sí**: lo que el agente hace decide qué experiencias vive |
| Analogía | Estudiar con el solucionario | Aprender a jugar un videojuego sin manual |

> 💡 La última fila es clave: si el ratón nunca va a la derecha, **nunca descubrirá que ahí hay queso**. Por eso el agente debe **explorar**, el tema de la Lección 14.2.

### ✋ Ejercicio 5 — ¿Es RL o no?
¿Cuáles de estos problemas se plantean naturalmente como **aprendizaje por refuerzo**?
- a) Predecir el precio de una casa con datos históricos de ventas.
- b) Un robot que aprende a caminar cayéndose y levantándose.
- c) Agrupar clientes por hábitos de compra.
- d) Decidir qué anuncio mostrar y aprender de si la gente hace clic.
- e) Un programa que aprende a jugar videojuegos de Atari solo mirando la pantalla y el marcador.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

**b), d) y e)**.
- a) es regresión supervisada (Lección 5.1): tienes la respuesta correcta (el precio).
- c) es no supervisado (Módulo 8): no hay acciones ni recompensas.
- b) y e) son RL de manual: acciones, consecuencias y una señal de premio.
- d) es un caso especial de RL llamado **bandido multibrazo** (Lección 14.2): cada decisión es independiente, pero solo ves el resultado del anuncio que **elegiste**, no de los demás.
</details>

---
## 4. 📖 ¿Qué pasa si diseñas mal la recompensa?

El agente optimiza **exactamente** la recompensa que le das, no la que *querías* darle. Si hay un atajo, lo encontrará.

> 🚤 **Caso real (OpenAI, 2016):** en el videojuego de carreras de lanchas *CoastRunners*, un agente recibía puntos por chocar objetivos en la pista. Descubrió que podía dar vueltas en círculo en una laguna, chocando los mismos objetivos una y otra vez, **sin terminar nunca la carrera**… y sacaba más puntos que cualquier humano.

A esto se le llama ***reward hacking*** (hacer trampa con la recompensa). Provoquémoslo en nuestro pasillo.

Imagina que un diseñador piensa: *"Quiero que el ratón sea activo; voy a darle **+1 por cada paso**"*. Probemos una cuarta política: el ratón que **va y viene** entre las casillas 3 y 4 sin llegar a ningún lado.

In [ ]:
def ir_y_volver(estado):
    return +1 if estado == 3 else -1      # de 3 va a 4, de 4 vuelve a 3... para siempre

### 🔮 Ejercicio 6 — El ratón tramposo
Con `costo_paso = +1` (premio por moverse) y un máximo de 50 pasos:
1. ¿Qué retorno obtiene **siempre derecha**?
2. ¿Y **ir y volver**?
3. ¿Cuál "prefiere" un agente que solo quiere maximizar el retorno?

✍️ *Mis predicciones:*

In [ ]:
pasillo_mal_disenado = Pasillo(costo_paso=+1)     # ✏️ prueba también con -1 y con 0

for nombre, pol in [("siempre derecha", siempre_derecha), ("ir y volver", ir_y_volver)]:
    print(f"{nombre:<16} → retorno = {jugar_episodio(pasillo_mal_disenado, pol)}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. Siempre derecha: +1 +1 +10 = **12**.
2. Ir y volver: 50 pasos × (+1) = **50**. Nunca termina por su cuenta; lo corta el límite de 50 pasos.
3. ¡Ir y volver! Para el agente, **pasear sin rumbo vale más que el queso**. No es un error del agente: es un error del **diseño de la recompensa**.

Con `costo_paso = -1`, ir y volver obtiene −50: la peor opción. El mismo agente, con otra recompensa, aprende un comportamiento totalmente distinto.
</details>

### 📖 Las perillas de diseño de un problema de RL
Antes de hablar de hiperparámetros del *agente* (Lección 14.2), quien diseña el *problema* ya tomó decisiones que lo cambian todo:

| Decisión de diseño | Qué controla | Si la subes ⬆️ | Si la bajas ⬇️ | Punto de partida |
|---|---|---|---|---|
| Costo por paso (`costo_paso`) | Urgencia por terminar | Positivo: premia alargar el episodio (¡peligro!) | Muy negativo: el agente prefiere terminar rápido aunque sea mal | Un pequeño negativo (−1) o 0 |
| Recompensa final (`premio_der`) | Qué tanto importa la meta | La meta domina sobre los costos del camino | Puede no compensar el camino | Mayor que el costo total de un camino razonable |
| Límite de pasos (`max_pasos`) | Duración máxima del episodio | Más tiempo para explorar, episodios más lentos | Puede cortar antes de llegar a la meta | Unas cuantas veces el largo de una buena solución |

### ✏️ Ejercicio 7 — Rediseña la recompensa
Cambia en la celda de abajo `costo_paso` a **−3** y ejecuta. Luego responde: ¿qué pasa con el ratón que va **siempre a la izquierda** (hacia el gato) comparado con un ratón que tarda mucho en llegar al queso? Para verlo, mira el retorno de **ir y volver** (que nunca llega a nada).

✍️ *Mi observación:*

In [ ]:
COSTO = -3   # ✏️ cambia este valor (prueba -1, -3, -5)

p = Pasillo(costo_paso=COSTO)
for nombre, pol in [("siempre derecha", siempre_derecha), ("siempre izquierda", siempre_izquierda),
                    ("ir y volver", ir_y_volver), ("al azar", al_azar)]:
    print(f"{nombre:<18} → retorno promedio = {retorno_promedio(p, pol, n=500):7.1f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Con `costo_paso = −3`:
- Siempre derecha: −3 −3 +10 = **4** (sigue siendo la mejor).
- Siempre izquierda: −3 −3 −10 = **−16**.
- Ir y volver: 50 × (−3) = **−150**.
- Al azar: cerca de **−25**, porque deambula muchos pasos.

Lanzarse al gato (−16) ya es **mucho mejor** que deambular (−150). Con costos muy altos por paso, un agente podría preferir terminar "mal pero rápido". El costo por paso es una forma de decirle al agente **cuánta prisa** tiene.
</details>

---
## 5. 📖 Retorno y descuento: ¿vale lo mismo un premio hoy que mañana?

Hasta ahora sumamos todas las recompensas igual. Pero en muchos problemas un premio **lejano** es menos seguro o menos valioso que uno **inmediato** (¿prefieres $100 hoy o $100 dentro de un año?).

Por eso en RL se usa el **retorno descontado**: cada recompensa futura se multiplica por un **factor de descuento** γ (gamma, entre 0 y 1) una vez por cada paso de espera:

```
retorno = r₁ + γ·r₂ + γ²·r₃ + γ³·r₄ + ...
```

- γ = 1 → todas las recompensas valen igual (lo que hicimos hasta ahora).
- γ = 0.9 → una recompensa a 10 pasos vale 0.9¹⁰ ≈ 35 % de su valor.
- γ = 0 → solo importa la recompensa inmediata (agente "miope").

Verás γ en detalle, como hiperparámetro, en la Lección 14.2.

### 🔮 Ejercicio 8 — Descuenta el queso
El ratón que va siempre a la derecha recibe las recompensas `[−1, −1, +10]`. Con γ = 0.9, ¿cuál es su retorno descontado? (Pista: −1 + 0.9·(−1) + 0.81·10.)

✍️ *Mi cálculo:*

In [ ]:
def retorno_descontado(recompensas, gamma):
    return sum(r * gamma**t for t, r in enumerate(recompensas))

for gamma in [1.0, 0.9, 0.5, 0.0]:     # ✏️ prueba otros valores
    print(f"γ = {gamma:<4} → retorno = {retorno_descontado([-1, -1, 10], gamma):6.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Con γ = 0.9: −1 − 0.9 + 8.1 = **6.2**. Con γ = 0.5 el queso casi no compensa el camino (**1.0**) y con γ = 0 el ratón solo ve el −1 inmediato (**−1**): es tan miope que **no le importa** el queso que está a dos pasos.
</details>

### 📖 El problema de la recompensa retrasada
En ajedrez, la recompensa (+1 ganar, −1 perder) llega **solo al final**, tras unas 40 jugadas propias. ¿Cuál de ellas fue la brillante y cuál el error? A esto se le llama el **problema de asignación de crédito** (*credit assignment*). Es parecido a la retropropagación (Lección 9.4), que reparte la culpa del error entre las neuronas, pero aquí se reparte **en el tiempo**, entre decisiones.

### ✋ Ejercicio 9 — Recompensas densas vs. escasas
Un robot debe aprender a salir de un laberinto. Dos diseñadores proponen:
- **A:** +100 al salir; 0 en cualquier otro paso.
- **B:** +100 al salir y, en cada paso, +1 si se acercó a la salida (en línea recta) y −1 si se alejó.

¿Con cuál aprenderá más rápido al principio? ¿Qué riesgo tiene B?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Con **B** suele aprender más rápido: recibe pistas en **cada paso** (recompensa **densa**), mientras que con A puede deambular miles de pasos sin recibir ninguna señal (recompensa **escasa**, *sparse*).

El riesgo de B: en un laberinto, a veces **hay que alejarse** de la salida para rodear una pared. Un agente que sigue la pista "acercarse en línea recta" puede quedarse atascado en un callejón sin salida que está cerca de la meta. Las recompensas "de ayuda" (*reward shaping*) aceleran el aprendizaje, pero pueden **enseñar atajos equivocados**: otra forma de *reward hacking*.
</details>

---
## 6. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| El agente hace algo raro que da muchos puntos pero no resuelve la tarea | *Reward hacking*: la recompensa no mide lo que quieres | Revisar la recompensa; observar episodios, no solo el número |
| El agente nunca aprende nada | Recompensa demasiado escasa: nunca la encuentra por azar | Recompensas intermedias con cuidado, episodios más cortos, más exploración (14.2) |
| El agente termina los episodios lo antes posible, aunque sea mal | Costo por paso demasiado alto | Bajar el castigo por paso |
| El agente alarga los episodios sin sentido | Recompensa positiva por paso o por "seguir vivo" | Quitarla o hacerla ≤ 0 |
| Comparas dos políticas con un solo episodio cada una | Ignoras el azar | Promediar cientos de episodios (y varias semillas) |

### 🔮 Ejercicio 10 — Diagnóstico rápido
Una empresa entrena un agente para un chat de soporte con la recompensa "**+1 por cada minuto que el cliente sigue en la conversación**". ¿Qué comportamiento indeseado podría aprender?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Podría aprender a **alargar las conversaciones** sin resolver nada: dar rodeos, hacer preguntas innecesarias o no dar nunca la solución. Es el "ir y volver" del ratón. Una recompensa mejor premiaría **problemas resueltos** (y quizá castigaría ligeramente el tiempo).
</details>

---
## 7. 📋 Ficha del concepto: el problema de RL

| | Aprendizaje por refuerzo |
|---|---|
| **Tipo** | Aprender a **actuar** mediante prueba y error |
| **Intuición** | Entrenar a un perro con premios |
| **Piezas** | Agente, entorno, estado, acción, recompensa, política, episodio |
| **Objetivo** | Maximizar el **retorno** (suma, normalmente descontada, de recompensas) |
| **Decisiones de diseño clave** | La recompensa (¡lo más delicado!), el costo por paso, el límite de pasos, γ |
| **✅ Fortalezas** | Resuelve problemas de decisiones en secuencia donde nadie conoce la respuesta correcta · Puede superar a los humanos (14.5) |
| **❌ Debilidades** | Necesita muchísima experiencia · Sensible al diseño de la recompensa · Difícil de depurar |
| **Úsalo cuando…** | Hay decisiones encadenadas, un simulador o mucha interacción, y una meta medible |
| **Evítalo cuando…** | Tienes ejemplos con la respuesta correcta (usa supervisado) o equivocarse durante el aprendizaje es caro o peligroso |

## 8. 🧠 Resumen en 5 puntos
1. En RL un **agente** aprende a actuar en un **entorno** probando **acciones** y recibiendo **recompensas**, sin que nadie le diga la respuesta correcta.
2. Una **política** es la regla del agente: qué acción tomar en cada estado. Aprender = mejorar la política.
3. El objetivo es el **retorno**: la suma de recompensas de todo el episodio, a menudo **descontada** con γ.
4. Las políticas con azar se comparan **simulando muchos episodios y promediando**.
5. El agente optimiza la recompensa que le das, no la que querías darle: cuidado con el ***reward hacking***.

---
## 9. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 9.1 Repaso rápido (opción múltiple)

**1.** ¿Qué diferencia esencial hay entre RL y aprendizaje supervisado?
- a) RL usa redes neuronales y supervisado no
- b) En RL el agente solo recibe una recompensa por lo que hizo, no la respuesta correcta
- c) RL no necesita datos
- d) Supervisado solo sirve para clasificar

**2.** La **política** de un agente es…
- a) La suma de recompensas · b) La regla que elige una acción en cada estado · c) El entorno · d) La recompensa máxima

**3.** Con γ = 0, el agente…
- a) Valora igual todas las recompensas · b) Solo se fija en la recompensa inmediata · c) No aprende nada · d) Prefiere las recompensas lejanas

**4.** Un agente de un videojuego descubre un error que le da puntos infinitos sin jugar. Esto es…
- a) Sobreajuste · b) Fuga de datos · c) *Reward hacking* · d) Exploración

**5.** Para saber si la política A es mejor que la B, cuando ambas tienen azar, lo correcto es…
- a) Jugar un episodio con cada una · b) Promediar el retorno de muchos episodios · c) Comparar la primera recompensa · d) Elegir la que termina antes

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Es la diferencia de fondo. RL puede usar o no redes neuronales (a), y sí necesita datos: los genera interactuando (c).
2. **b)**
3. **b)** El agente miope del Ejercicio 8.
4. **c)** Como la lancha de *CoastRunners*.
5. **b)** Un solo episodio puede ser suerte.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** ¿Por qué entrenar con RL un auto autónomo **directamente en la calle** sería mala idea? ¿Qué harías en su lugar?

**B. Caso de análisis 💼** Una app de ejercicio quiere usar RL para decidir **cuándo enviar notificaciones**. El equipo propone la recompensa "+1 cada vez que el usuario abre la app". Escribe un riesgo de esa recompensa y propón una mejor.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Porque el agente aprende **equivocándose**, y aquí cada error puede ser un accidente. Se entrena primero en un **simulador** (millones de kilómetros virtuales), se combina con datos de conductores humanos y solo después se prueba con mucha supervisión. Volverás a esta idea en robótica (Lección 14.5).

**B.** Riesgo: el agente puede aprender a **bombardear** con notificaciones (más notificaciones → más aperturas a corto plazo) hasta que el usuario se harte y desinstale la app. Una recompensa mejor mira el **largo plazo**: minutos de ejercicio a la semana, o retención a 30 días, y castiga las desinstalaciones o las notificaciones silenciadas.
</details>

### 9.3 🧩 Práctica integradora: "¿Queso chico o queso grande?"

Ahora el pasillo tiene **dos quesos**: uno **chico** (+2) en la casilla 0 y uno **grande** (+10) en la casilla 6. El ratón empieza en la casilla **1**, justo al lado del queso chico.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Construye el entorno**

In [ ]:
dos_quesos = Pasillo(inicio=1, premio_izq=2, premio_der=10, costo_paso=-1)
print(dos_quesos.reiniciar(), dos_quesos.dibujar())

**Paso 2 🔮 — Calcula a mano.** Con `costo_paso = −1`, ¿qué retorno obtiene ir **siempre a la izquierda** (queso chico) y ir **siempre a la derecha** (queso grande)?

✍️ *Mi cálculo:*

In [ ]:
for nombre, pol in [("siempre izquierda", siempre_izquierda), ("siempre derecha", siempre_derecha)]:
    print(f"{nombre:<18} → retorno = {jugar_episodio(dos_quesos, pol)}")

<details><summary><b>👀 Ver respuesta</b></summary>

- Izquierda: un solo paso y llega → **+2**.
- Derecha: 4 pasos normales (−4) y luego +10 → **+6**.

Con este costo por paso, **vale la pena caminar** hasta el queso grande.
</details>

**Paso 3 ✏️ — Sube el costo del camino.** Cambia `COSTO` a −3. ¿Sigue conviniendo el queso grande?

In [ ]:
COSTO = -3   # ✏️
dos_quesos = Pasillo(inicio=1, premio_izq=2, premio_der=10, costo_paso=COSTO)
for nombre, pol in [("siempre izquierda", siempre_izquierda), ("siempre derecha", siempre_derecha)]:
    print(f"{nombre:<18} → retorno = {jugar_episodio(dos_quesos, pol)}")

<details><summary><b>👀 Ver respuesta</b></summary>

Con −3: derecha = 4·(−3) + 10 = **−2**, izquierda = **+2**. Ahora gana el **queso chico**: el camino largo cuesta más de lo que vale el premio.
</details>

**Paso 4 ✋ — Encuentra el punto de empate.** ¿Con qué `costo_paso` dan **exactamente lo mismo** las dos políticas? Resuélvelo en papel (izquierda siempre vale +2; derecha vale 4·costo + 10) y compruébalo con la celda de arriba.

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

4·costo + 10 = 2 → costo = **−2**. Con `COSTO = -2` ambas dan +2. Con costos más suaves (−1, −0.5…) conviene el queso grande; con costos más duros, el chico.

Esta es la esencia de muchas decisiones de RL (y de la vida): **¿premio pequeño seguro y cercano, o grande y lejano?** La respuesta depende de cómo diseñes las recompensas.
</details>

**Paso 5 ✋ — Piensa como diseñador.** Quieres que el ratón aprenda a ir al queso **grande**, pero el costo de cada paso es −3 (es un valor real: el ratón gasta energía). Sin tocar `costo_paso`, ¿qué otra perilla del entorno o del agente podrías cambiar? Escribe dos opciones.

✍️ *Mis opciones:*

<details><summary><b>👀 Ver respuesta</b></summary>

Algunas opciones válidas:
1. **Subir el premio del queso grande** (`premio_der`), por ejemplo a 20: derecha = −12 + 20 = 8 > 2.
2. **Bajar el premio del chico** o quitarlo.
3. **Empezar más cerca** del queso grande (`inicio`).
4. (Recuerda el Ejercicio 8) Con **descuento** γ < 1 pasa lo contrario: el queso lejano vale todavía **menos**. Si quieres un agente "paciente", necesitas un γ cercano a 1.

Fíjate en que **todas** estas opciones cambian el comportamiento sin tocar el algoritmo de aprendizaje: en RL, diseñar el problema es la mitad del trabajo.
</details>

---
## 📚 Para profundizar
- 📘 **Libro (gratuito en línea):** Sutton y Barto, *Reinforcement Learning: An Introduction* (2.ª ed.), capítulos 1 y 3. Es "la biblia" del RL.
- 📺 **Video:** DeepMind x UCL, *Reinforcement Learning Lecture Series*, clase 1 (inglés).
- 🎮 **Interactivo:** biblioteca **Gymnasium** (antes OpenAI Gym): entornos clásicos listos para usar, con la misma interfaz `reset()` / `step()` que nuestro `Pasillo`.
- 📰 **Lectura:** OpenAI, *Faulty Reward Functions in the Wild* (2016), el caso de *CoastRunners*.

---
⬅️ **Anterior:** [13.10 Proyecto del módulo: El asistente de Cóndor Air ✈️](../m13-generativos-llms/13.10-proyecto-del-modulo.md) · ➡️ **Siguiente:** [14.2 Exploración vs. explotación: ¿lo de siempre o algo nuevo?](14.2-exploracion-vs-explotacion.ipynb)